In [11]:
# ============================================================
# Moving Mirror Dataset + FNO Training (Davies + Good)
# ============================================================

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from neuralop.models import FNO
from scipy.special import loggamma
import mpmath as mp

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# Grids
# ------------------------------------------------------------
N_t = 128
t = np.linspace(0.0, 10.0, N_t)

N_w = 30
omega = np.linspace(0.01, 2.0, N_w)
omega_p = np.linspace(0.01, 2.0, N_w)

# ------------------------------------------------------------
# Davies trajectory
# ------------------------------------------------------------
def z_davies(t, A, B, kappa):
    return -t - A*np.exp(-2*kappa*t) + B

# ------------------------------------------------------------
# Good trajectory (solve implicit equation)
# ------------------------------------------------------------
def solve_x(t, kappa, g):
    x = -t  # initial guess
    for _ in range(20):
        f = g*(t + x) + np.sinh(2*kappa*x)
        df = g + 2*kappa*np.cosh(2*kappa*x)
        x = x - f/df
    return x

def z_good(t, kappa, g=1.0):
    return np.array([solve_x(tt, kappa, g) for tt in t])

# ------------------------------------------------------------
# Davies beta
# ------------------------------------------------------------
def beta_davies(omega, omega_p, A, B, kappa):
    D = (1/kappa)*np.log(A) - B

    omega_ = omega[:,None]
    omega_p_ = omega_p[None,:]

    log_pref = (
        -np.log(2*np.pi)
        -0.5*(np.log(omega_) + np.log(omega_p_))
        -np.pi*omega_p_/(2*kappa)
    )

    phase = (
        -1j*omega_p_*D
        +1j*omega_*B
        -1j*(omega_p_/kappa)*np.log(omega_)
    )

    lgamma = loggamma(1 + 1j*omega_p_/kappa)
    log_beta = log_pref + phase + lgamma

    beta = -1j*np.exp(np.clip(log_beta.real,-700,700)) * np.exp(1j*log_beta.imag)
    return beta

# ------------------------------------------------------------
# Good beta (mpmath)
# ------------------------------------------------------------
def beta_good(omega, omega_p, kappa):
    beta = np.zeros((len(omega), len(omega_p)), dtype=np.complex128)

    for i,w in enumerate(omega):
        for j,wp in enumerate(omega_p):
            pref = -np.sqrt(w*wp)/(np.pi*kappa*wp)
            thermal = np.exp(-np.pi*w/(2*kappa))
            K = mp.besselk(1j*w/kappa, wp/kappa)
            beta[i,j] = pref * thermal * complex(K)

    return beta

# ------------------------------------------------------------
# Dataset generation
# ------------------------------------------------------------
def generate_dataset(n_samples):
    X, Y = [], []

    for _ in range(n_samples):

        if np.random.rand() < 0.5:
            # Davies
            A = np.random.uniform(0.5, 1.5)
            B = np.random.uniform(0.0, 1.0)
            kappa = np.random.uniform(0.5, 1.5)

            z = z_davies(t, A, B, kappa)
            beta = beta_davies(omega, omega_p, A, B, kappa)

        else:
            # Good
            kappa = np.random.uniform(0.5, 1.5)

            z = z_good(t, kappa)
            beta = beta_good(omega, omega_p, kappa)

        # input channels (broadcast z(t))
        channels = []
        for val in z:
            channels.append(np.full((N_w, N_w), val))

        # add coordinate channels
        omega_grid = np.tile(omega[:,None], (1,N_w))
        omega_p_grid = np.tile(omega_p[None,:], (N_w,1))

        channels.append(omega_grid)
        channels.append(omega_p_grid)

        X.append(np.stack(channels, axis=0))
        Y.append(np.stack([beta.real, beta.imag], axis=0))

    return torch.tensor(np.array(X),dtype=torch.float32), \
           torch.tensor(np.array(Y),dtype=torch.float32)

# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------
X_train, Y_train = generate_dataset(300)
X_val, Y_val = generate_dataset(60)

train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=8, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, Y_val), batch_size=8)

# ------------------------------------------------------------
# FNO model
# ------------------------------------------------------------
model = FNO(
    n_modes=(12,12),
    hidden_channels=64,
    in_channels=N_t + 2,
    out_channels=2,
    n_layers=4
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# ------------------------------------------------------------
# Training loop (with validation)
# ------------------------------------------------------------
for epoch in range(20):

    # ---- TRAIN ----
    model.train()
    train_loss = 0

    for xb,yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ---- VALIDATION ----
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for xb,yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            val_loss += loss_fn(pred, yb).item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch:02d} | Train {train_loss:.4e} | Val {val_loss:.4e}")

Epoch 00 | Train 1.4238e-01 | Val 1.3687e-01
Epoch 01 | Train 1.0210e-01 | Val 7.2427e-02
Epoch 02 | Train 5.8202e-02 | Val 4.0783e-02
Epoch 03 | Train 2.2237e-02 | Val 8.5698e-03
Epoch 04 | Train 6.8304e-03 | Val 6.9976e-03
Epoch 05 | Train 5.1455e-03 | Val 5.3125e-03
Epoch 06 | Train 4.5955e-03 | Val 5.2560e-03
Epoch 07 | Train 3.8357e-03 | Val 3.6041e-03
Epoch 08 | Train 3.4742e-03 | Val 3.7650e-03
Epoch 09 | Train 3.1200e-03 | Val 3.7005e-03
Epoch 10 | Train 2.9046e-03 | Val 2.7556e-03
Epoch 11 | Train 2.8092e-03 | Val 2.8696e-03
Epoch 12 | Train 2.6890e-03 | Val 2.7570e-03
Epoch 13 | Train 2.6284e-03 | Val 2.7959e-03
Epoch 14 | Train 2.3996e-03 | Val 2.4944e-03
Epoch 15 | Train 2.5546e-03 | Val 2.3610e-03
Epoch 16 | Train 2.2100e-03 | Val 2.5592e-03
Epoch 17 | Train 2.1669e-03 | Val 2.6094e-03
Epoch 18 | Train 2.3053e-03 | Val 2.1191e-03
Epoch 19 | Train 1.9675e-03 | Val 2.4073e-03
